In [ ]:
import sys, os, pickle, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
from spk_lfp_cluster_comp_analysis import *
from config import SPE1_PICKLE_ROOT, PRIORITY_CELLS, CELL_IDS

warnings.filterwarnings("ignore")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
SLIDING_DIR  = os.path.join(SPE1_PICKLE_ROOT, "lfp_spk_group_pickles")
PREPOST_DIR  = os.path.join(SPE1_PICKLE_ROOT, "prepost_specparam_pickles")
CLUSTER_DIR  = os.path.join(SPE1_PICKLE_ROOT, "cluster_pickles")

# ── Cell subsets ───────────────────────────────────────────────────────────────
subsets = get_cell_subsets(PRIORITY_CELLS, CELL_IDS)
# subsets["priority"] — priority cells only
# subsets["np"]       — non-priority cells
# subsets["all"]      — all cells combined
print(f"Priority: {len(subsets['priority'])} cells")
print(f"NP:       {len(subsets['np'])} cells")
print(f"All:      {len(subsets['all'])} cells")
# Spike feature colour palette (consistent across all notebooks)
feature_shades = {
    'peak_amp_cluster':         '#8c564b',
    'peak_sharpness_cluster':   '#a06d62',
    'peak_width_cluster':       '#b38479',
    'exp_lambda_cluster':       '#c561a8',
    'inflection_time_cluster':  '#9b59b6',
    'exp_const_cluster':        '#d7aee0',
    'log_isi_cluster':          '#7f7f7f',
    'spk_times_ms_cluster':     '#b0b0b0',
}


## 1. Load Data

In [ ]:
df_pop_stats, pop_traces = compile_lfp_stats(SLIDING_DIR)
df_prepost               = compile_prepost_stats(PREPOST_DIR)

# Load spike cluster master table
from spk_feat_cluster_comp_analysis import compile_experiment_results
df_master = compile_experiment_results(CLUSTER_DIR)
print(f"Sliding stats: {df_pop_stats['cell_id'].nunique()} cells")
print(f"Pre/post:      {df_prepost['cell_id'].nunique()} cells")
print(f"Cluster master: {df_master['cell_id'].nunique()} cells")

In [ ]:
SUBSET_LABELS = [
    ("Priority cells",      subsets["priority"]),
    ("Non-priority cells",  subsets["np"]),
    ("All cells combined",  subsets["all"]),
]

## 2. Waveform Difference vs LFP Effect Strength

Do cells with bigger spike waveform cluster differences show stronger LFP modulation?

In [ ]:
from scipy.stats import spearmanr

for label, cell_ids in SUBSET_LABELS:
    print(f"\n{'='*60}\n{label}\n{'='*60}")
    df_sub    = filter_df_stats(df_pop_stats, cell_ids)
    df_merged = df_sub.merge(
        df_master[["cell_id", "spike_feature", "nRMSE", "cos_sim"]].drop_duplicates(),
        on=["cell_id", "spike_feature"], how="left")

    metrics = {"nRMSE": "Waveform nRMSE", "cos_sim": "Cosine Similarity"}
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"Waveform Difference vs LFP Effect — {label}", fontsize=13, fontweight="bold")
    for ax, (col, col_label) in zip(axes, metrics.items()):
        d = df_merged.dropna(subset=[col, "cohens_d"])
        if d.empty: continue
        r, p = spearmanr(d[col], d["cohens_d"].abs())
        ax.scatter(d[col], d["cohens_d"].abs(), alpha=0.3, s=15)
        ax.set_xlabel(col_label); ax.set_ylabel("|Cohen's d|")
        ax.set_title(f"ρ={r:.2f}, p={p:.3f}")
        sns.despine(ax=ax)
    plt.tight_layout(); plt.show()

## 3. Cell Metadata vs LFP Effect Strength

Does recording modality, cell type, or cortical depth predict LFP effect?

In [ ]:
from config import DICT_CELL_TYPE, DICT_PATCH_TYPE, DICT_CORT_DEPTH
from scipy.stats import kruskal

for label, cell_ids in SUBSET_LABELS:
    print(f"\n{'='*60}\n{label}\n{'='*60}")
    df_sub = filter_df_stats(df_pop_stats, cell_ids)
    df_sub = df_sub.copy()
    df_sub["cell_num"]   = df_sub["cell_id"].str.lstrip("c").astype(int)
    df_sub["cell_type"]  = df_sub["cell_num"].map(DICT_CELL_TYPE)
    df_sub["patch_type"] = df_sub["cell_num"].map(DICT_PATCH_TYPE)
    df_sub["cort_depth"] = df_sub["cell_num"].map(DICT_CORT_DEPTH)

    cat_vars = ["cell_type", "patch_type"]
    fig, axes = plt.subplots(1, len(cat_vars), figsize=(6 * len(cat_vars), 4))
    fig.suptitle(f"Metadata vs |Cohen's d| — {label}", fontsize=13, fontweight="bold")
    for ax, var in zip(np.array(axes).flatten(), cat_vars):
        d = df_sub.dropna(subset=[var, "cohens_d"])
        groups = {g: d.loc[d[var] == g, "cohens_d"].abs() for g in d[var].unique()}
        if len(groups) >= 2:
            stat, p = kruskal(*groups.values())
            ax.set_title(f"{var}  (Kruskal p={p:.3f})")
        sns.boxplot(data=d, x=var, y=d["cohens_d"].abs(), ax=ax)
        ax.set_ylabel("|Cohen's d|"); ax.set_xlabel("")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
        sns.despine(ax=ax)
    plt.tight_layout(); plt.show()

## 4. Pre/Post Effect vs Metadata

In [ ]:
for label, cell_ids in SUBSET_LABELS:
    print(f"\n{'='*60}\n{label}\n{'='*60}")
    df_sub = df_prepost[df_prepost["cell_id"].isin(cell_ids)].copy()
    df_sub["cell_num"]  = df_sub["cell_id"].str.lstrip("c").astype(int)
    df_sub["cell_type"] = df_sub["cell_num"].map(DICT_CELL_TYPE)

    df_w = df_sub[df_sub["window"] == "within"].dropna(subset=["cohens_dz", "cell_type"])
    if df_w.empty: continue
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.boxplot(data=df_w, x="cell_type", y="cohens_dz", ax=ax)
    ax.axhline(0, color="gray", ls="--", lw=0.8)
    ax.set_title(f"Pre→Post d_z by Cell Type — {label}", fontsize=12)
    ax.set_xlabel("Cell Type"); ax.set_ylabel("Cohen's d_z")
    sns.despine(ax=ax)
    plt.tight_layout(); plt.show()